In [ ]:
import pandas as pd
import numpy as np

#################################################
# Create random data for multiple accounts
# - To be ingested from an excel
#################################################

data = {
    'Account ID': ['Account001', 'Account002', 'Account003'],
    'Client Name': ['Alice Smith', 'Bob Johnson', 'Charlie Brown'],
    'Balance': [1500.75, 300.50, 5000.00],
    'Key Stock Holding': ['Microsoft', 'Google', 'Intel']
}



# Create the pandas DataFrame
client_accounts_df = pd.DataFrame(data)

# Unique stock holdings
unique_stocks = client_accounts_df['Key Stock Holding'].unique()

# Add new performance columns with random numbers
np.random.seed(42) # for reproducibility
client_accounts_df['Yesterday\'s Performance'] = np.round(np.random.uniform(-0.5, 0.5, len(client_accounts_df)), 2)
client_accounts_df['Past 1 Month Performance'] = np.round(np.random.uniform(-5, 5, len(client_accounts_df)), 2)
client_accounts_df['Year to Date Performance'] = np.round(np.random.uniform(-10, 15, len(client_accounts_df)), 2)

# Display the DataFrame with new columns
print(client_accounts_df)
print(unique_stocks)

   Account ID    Client Name  Balance Key Stock Holding  \
0  Account001    Alice Smith  1500.75         Microsoft   
1  Account002    Bob Johnson   300.50            Google   
2  Account003  Charlie Brown  5000.00             Intel   

   Yesterday's Performance  Past 1 Month Performance  Year to Date Performance  
0                    -0.13                      0.99                     -8.55  
1                     0.45                     -3.44                     11.65  
2                     0.23                     -3.44                      5.03  
['Microsoft' 'Google' 'Intel']


In [ ]:
!pip install yfinance

import yfinance as yf
import pandas as pd

# Define the tickers for DOW, S&P500, and NASDAQ
tickers = ['^DJI', '^GSPC', '^NDX']
# DJI = Dow Jones
# GSPC = S&P500
# NDX = NASDAQ


# Fetch historical data for the last 2 days (to calculate previous day's return)
# period='2d' will fetch data for today and yesterday if available, or just enough to calculate 1 day return
hist_data = yf.download(tickers, period='5d')

# Calculate previous day's returns
# We'll use the 'Close' prices for calculation
returns = hist_data['Close'].pct_change().iloc[-1]*100

# Extract the latest trading date from the 'returns' Series index
report_date = returns.name.strftime('%Y-%m-%d')

print("Previous Day's Index Returns:")
print(returns)
print('Report Date: ',report_date)

/tmp/ipykernel_904/3143348707.py:15: FutureWarning: YF.download() has changed argument auto_adjust default to True
  hist_data = yf.download(tickers, period='5d')
[*********************100%***********************]  3 of 3 completed

Previous Day's Index Returns:
Ticker
^DJI     1.788254
^GSPC    1.204046
^NDX     1.288990
Name: 2026-04-17 00:00:00, dtype: float64
Report Date:  2026-04-17


In [ ]:
import requests
from google.colab import userdata
from datetime import date
import pandas as pd # Ensure pandas is imported for pd.to_datetime

p1d_date = (pd.to_datetime(report_date) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
p3d_date = (pd.to_datetime(report_date) - pd.Timedelta(days=3)).strftime('%Y-%m-%d')
p7d_date = (pd.to_datetime(report_date) - pd.Timedelta(days=30)).strftime('%Y-%m-%d')

# Format to take news in past 1 week. Pull all required tickers to reduce API calls.
from_date = p7d_date

print(f"Using date: {from_date}")


# Set message to key value pair in dictionary.
msg_dict = {}

# IMPORTANT: Replace 'YOUR_NEWSAPI_KEY' with your actual NewsAPI key.
# You can get one by signing up at NewsAPI.org.
# If using Colab Secrets, make sure to add your API key under the name 'NEWS_API_KEY'
NEWS_API_KEY = userdata.get('NEWS_API_KEY')
new_list = []
for stock in unique_stocks:

  msg = "" #Initiailize an empty message.

  # Define the search query and parameters
  query = f'{stock} -stock -options' #Remove stock and options in title to reduce low quality articles
  language = 'en' # English articles
  sort_by = 'publishedAt' # Sort by relevancy / publishedAt
  page_size = 10 # Number of articles to fetch

  # Construct the API endpoint URL
  # For top headlines: https://newsapi.org/v2/top-headlines
  # For all articles: https://newsapi.org/v2/everything
  # q = query whole article / qInTitle = query in title only
  url = f'https://newsapi.org/v2/everything?qInTitle={query}&from={from_date}&language={language}&sortBy={sort_by}&pageSize={page_size}&apiKey={NEWS_API_KEY}'




  try:
      response = requests.get(url)
      response.raise_for_status() # Raise an exception for HTTP errors
      data = response.json()

      if data['status'] == 'ok':
          articles = data['articles']
          if articles:
              print(f"Found {len(articles)} articles about '{query}' from NewsAPI:")
              for i, article in enumerate(articles):

                  new_list += [{article.get('source', {}).get('name')}]

                  if i >100: #If more than 3 articles in the past week, not necessary to return the rest
                    continue
                  # elif upper(article.get('title')) not in ("YAHOO ENTERTAINMENT","REUTERS",""):

                  print(f"\nArticle {i+1}:")
                  print(f"  Title: {article.get('title')}")
                  print(f"  Source: {article.get('source', {}).get('name')}")
                  print(f"  Date: {pd.to_datetime(article.get('publishedAt')).strftime('%Y-%m-%d')}")
                  print(f"  URL: {article.get('url')}")
                  # print(f"  Description: {article.get('description')}")

                  # Corrected line to format the date
                  msg += f"\n Article {i+1}: {article.get('title')} \n URL: {article.get('url')} \n Source:{article.get('source', {}).get('name')} \n Date: {pd.to_datetime(article.get('publishedAt')).strftime('%Y-%m-%d')} \n"
          else:
              msg = ""
              print(f"No articles found for '{query}' using NewsAPI.")
      else:
          print(f"NewsAPI Error: {data.get('message', 'Unknown error')}")

      msg_dict[f'{stock}'] = msg

  except requests.exceptions.RequestException as e:
      print(f"Error fetching news from NewsAPI: {e}")
  except Exception as e:
      print(f"An unexpected error occurred: {e}")

Using date: 2026-03-18
Error fetching news from NewsAPI: 426 Client Error: Upgrade Required for url: https://newsapi.org/v2/everything?qInTitle=Microsoft%20-stock%20-options&from=2026-03-18&language=en&sortBy=publishedAt&pageSize=10&apiKey=ed7cf75e42974d10ab67e3fa41e3a4e7
Error fetching news from NewsAPI: 426 Client Error: Upgrade Required for url: https://newsapi.org/v2/everything?qInTitle=Google%20-stock%20-options&from=2026-03-18&language=en&sortBy=publishedAt&pageSize=10&apiKey=ed7cf75e42974d10ab67e3fa41e3a4e7
Error fetching news from NewsAPI: 426 Client Error: Upgrade Required for url: https://newsapi.org/v2/everything?qInTitle=Intel%20-stock%20-options&from=2026-03-18&language=en&sortBy=publishedAt&pageSize=10&apiKey=ed7cf75e42974d10ab67e3fa41e3a4e7


In [ ]:
new_list

[]

In [ ]:
#######################################################
# Pull Headlines from Reuters - Via google search
#######################################################

!pip install feedparser

import feedparser
from urllib.parse import urlparse
import textwrap # Import the textwrap module

# Reuters Top News RSS Feed URL (using the new URL provided by the user)
RSS_FEED_URL = 'https://news.google.com/rss/search?q=site%3Areuters.com&hl=en-US&gl=US&ceid=US%3Aen'

# Parse the RSS feed
feed = feedparser.parse(RSS_FEED_URL) # Corrected to use the new RSS_FEED_URL

# Headlines to be used for all clients
headlines = ""

# Check for any errors during parsing
if feed.bozo: # bozo is true if feed is malformed
    print(f"Error parsing feed: {feed.bozo_exception}")
else:
    print(f"--- Latest Reuters Headlines ---")
    # Iterate through the entries (articles) in the feed
    for i, entry in enumerate(feed.entries):
        if i >= 3: # Limit to top 3 headlines for brevity
            break
        print(f"\nHeadline {i+1}:")
        print(f"  Title: {entry.title}")

        # Use textwrap to format the link for display
        wrapped_link = textwrap.fill(entry.link, width=70, subsequent_indent='        ')
        print(f"  Link: {entry.link}")

        # The 'headlines' variable still stores the full URL
        headlines += f"\nHeadline {i+1}: {entry.title} \n Link: {entry.link} \n"

        # Some feeds might have 'published' or 'updated' instead of 'published_parsed'
        if hasattr(entry, 'published_parsed'):
            # Format the date nicely
            published_date = f"{entry.published_parsed.tm_year}-{entry.published_parsed.tm_mon:02d}-{entry.published_parsed.tm_mday:02d}"
            print(f"  Date: {published_date}")
            # headlines += f"  Date: {published_date} \n"

        elif hasattr(entry, 'updated_parsed'):
            updated_date = f"{entry.updated_parsed.tm_year}-{entry.updated_parsed.tm_mon:02d}-{entry.updated_parsed.tm_mday:02d}"
            print(f"  Date: {updated_date}")
            # headlines += f"  Date: {updated_date} \n"

--- Latest Reuters Headlines ---

Headline 1:
  Title: Apple withholds data in India antitrust case, watchdog sets final hearing - Reuters
  Link: https://news.google.com/rss/articles/CBMi3gFBVV95cUxONGY5WnNXcDNfUlQ5MU4wRmdlLTlJSnFnTkVsanZwNFllWERVV1V5SlNJUkF1MENIcTFfUHlHZ3BENVN6ckhzWUNEeTM0OC1kUFNpUUpsblhmVjE4aTNXLWZZWHdHbE1LaWZhN1hVTkRkd0gzYk5tM2ZQZEJEX09zZk1lcVF2ejFzLV84WHVhbVM2RHJ0cThOdm9fRzdpUFJGeWtpMDhjN0Q5MUdlTHpDSG9ib0NmOFViUmVFeGRyQkZ6aEFrdWxfRERZemo1WmF3TGVLVk5NR2lLWktJb2c?oc=5
  Date: 2026-04-20

Headline 2:
  Title: Major 7.5-magnitude quake hits off Japan, triggers tsunami warnings - Reuters
  Link: https://news.google.com/rss/articles/CBMivAFBVV95cUxQMUhrZE8yWnRQNzhsZG1na2tQTVlQYmlOcFR2elRyVEw3UDdpTktrbDJ1bUNsWm40dEYybDVzTWlrb2xnckg0ME5yalFQRDdYVE5QUVg1TWJmSkFmeVg3QmZ4Wk8xbDN0WElGOXgtU2hJemdRSVkxaklnb0w2SHRFbVdqVGlONVh3bW8yWFd1Uk11STZwczIySmU3dksxczNORUlUVDh5cUkzUThiQUkydTlzNVNqbTV3NmliRQ?oc=5
  Date: 2026-04-20

Headline 3:
  Title: China voices concern over US seizure o

In [ ]:
headlines

'\nHeadline 1: Apple withholds data in India antitrust case, watchdog sets final hearing - Reuters \n Link: https://news.google.com/rss/articles/CBMi3gFBVV95cUxONGY5WnNXcDNfUlQ5MU4wRmdlLTlJSnFnTkVsanZwNFllWERVV1V5SlNJUkF1MENIcTFfUHlHZ3BENVN6ckhzWUNEeTM0OC1kUFNpUUpsblhmVjE4aTNXLWZZWHdHbE1LaWZhN1hVTkRkd0gzYk5tM2ZQZEJEX09zZk1lcVF2ejFzLV84WHVhbVM2RHJ0cThOdm9fRzdpUFJGeWtpMDhjN0Q5MUdlTHpDSG9ib0NmOFViUmVFeGRyQkZ6aEFrdWxfRERZemo1WmF3TGVLVk5NR2lLWktJb2c?oc=5 \n\nHeadline 2: Major 7.5-magnitude quake hits off Japan, triggers tsunami warnings - Reuters \n Link: https://news.google.com/rss/articles/CBMivAFBVV95cUxQMUhrZE8yWnRQNzhsZG1na2tQTVlQYmlOcFR2elRyVEw3UDdpTktrbDJ1bUNsWm40dEYybDVzTWlrb2xnckg0ME5yalFQRDdYVE5QUVg1TWJmSkFmeVg3QmZ4Wk8xbDN0WElGOXgtU2hJemdRSVkxaklnb0w2SHRFbVdqVGlONVh3bW8yWFd1Uk11STZwczIySmU3dksxczNORUlUVDh5cUkzUThiQUkydTlzNVNqbTV3NmliRQ?oc=5 \n\nHeadline 3: China voices concern over US seizure of Iranian cargo ship, urges further talks - Reuters \n Link: https://news.google.com/rss

In [ ]:
#######################################################
# use Gemini Model to generate LLM output using prompt
# Augment answer with latest market news
#######################################################

import google.generativeai as genai
import os
from google.colab import userdata

# Configure Gemini API (replace 'YOUR_API_KEY' if not using Colab's secret manager)
# In Google Colab, you can often access API keys securely via 'Secrets' (the key icon on the left).
# If running outside Colab or if you prefer direct input, uncomment the line below and set your key.
# genai.configure(api_key="YOUR_API_KEY")

# If using Colab Secrets, make sure to add your API key under the name 'GEMINI_API_KEY'
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GEMINI_API_KEY)

# Initialize the Gemini model
model = genai.GenerativeModel('gemini-2.5-flash') # Changed from 'gemini-pro'

# Extract the latest trading date from the 'returns' Series index
report_date = returns.name.strftime('%Y-%m-%d')



# List to store generated reports
generated_reports = []

# Iterate through each client account and generate a write-up using Gemini
for index, client in client_accounts_df.iterrows():
    client_name = client['Client Name']
    account_id = client['Account ID']
    # The 'Account Type' column was removed from client_accounts_df in a previous step
    # account_type = client['Account Type']
    balance = client['Balance']
    # The original Last Activity was removed from client_accounts_df, so commenting this out
    # last_activity = client['Last Activity']
    yesterday_performance = client['Yesterday\'s Performance']
    month_performance = client['Past 1 Month Performance']
    ytd_performance = client['Year to Date Performance']
    key_stock_holding = client['Key Stock Holding'] # Get the key stock holding

    market_overview_str = "Market Overview (Previous Day's Returns):\n"
    for ticker, ret in returns.items():
        market_overview_str += f"  {ticker}: {ret:+.4f}%\n"

    prompt = f"""Generate a personalized financial report for the following client:

Date: {report_date}
Client Name: {client_name}
Account ID: {account_id}
Current Balance: ${balance:,.2f}
Yesterday's Account Performance: {yesterday_performance:+.2f}%
Past 1 Month Account Performance: {month_performance:+.2f}%
Year to Date Account Performance: {ytd_performance:+.2f}%


{market_overview_str}

Provide a concise and professional summary, highlighting key performance metrics and mentioning the general market trend from the provided market overview, using the returns generated from dataframe 'returns', set all performance numbers to 1 decimal place. Combine both the summary of the account and the market indexes

Do not make any investment recommendations or future predictions.

Remove all the stars from the formatting as this will be in clean text, unless it is the title of the header.

"""

# Relevant news articles in the past week for portfolio:
# {msg_dict[f'{key_stock_holding}']}


    print(f"--- Generating Report for {client_name} (Account ID: {account_id}) ---")
    try:
        response = model.generate_content(prompt)
        # Extract the text content from the GenerateContentResponse object before concatenation
        report_content = response.text + f"\nLatest News Headlines: \n{headlines}"

        generated_reports.append(report_content) # Store the report text
        print(report_content)
        print("\n" + "="*80 + "\n") # Separator for readability
    except Exception as e:
        generated_reports.append(f"Error generating report: {e} \n Latest News Headlines: \n {headlines}") # Store error message
        print(f"Error generating report for {client_name}: {e}")
        print("\n" + "="*80 + "\n")

# Add the generated reports as a new column to the DataFrame
client_accounts_df['Financial Report'] = generated_reports

# Display the DataFrame with the new column
# display(client_accounts_df)

--- Generating Report for Alice Smith (Account ID: Account001) ---


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1353.60ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1467.16ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1479.85ms


Financial Report for Alice Smith
Date: April 17, 2026

Dear Alice Smith,

This report provides a summary of your account performance (Account ID: Account001) as of April 17, 2026.

Your current balance is $1,500.75. Regarding your account's performance, yesterday's return was -0.1%. Over the past month, your account generated a return of +1.0%, while the year-to-date performance stands at -8.6%.

In the broader market, yesterday saw positive movements across major indexes. The Dow Jones Industrial Average (^DJI) rose by +1.8%, the S&P 500 (^GSPC) increased by +1.2%, and the NASDAQ 100 (^NDX) was up by +1.3%.

This report is for informational purposes only.
Latest News Headlines: 

Headline 1: Apple withholds data in India antitrust case, watchdog sets final hearing - Reuters 
 Link: https://news.google.com/rss/articles/CBMi3gFBVV95cUxONGY5WnNXcDNfUlQ5MU4wRmdlLTlJSnFnTkVsanZwNFllWERVV1V5SlNJUkF1MENIcTFfUHlHZ3BENVN6ckhzWUNEeTM0OC1kUFNpUUpsblhmVjE4aTNXLWZZWHdHbE1LaWZhN1hVTkRkd0gzYk5tM2ZQZ

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1088.31ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 7274.35ms


Error generating report for Bob Johnson: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 1.936760086s.


--- Generating Report for Charlie Brown (Account ID: Account003) ---


Error generating report for Charlie Brown: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash
Please retry in 1.423203042s.




In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Define the path to save the results in your Google Drive
# You can change 'MyDrive/Colab Notebooks/' to any folder within your Drive
save_path = f'/content/drive/MyDrive/Daily_LLM_Summary_{report_date}.csv'


# Save the results_df to CSV
client_accounts_df.to_csv(save_path, index=False)


print(f"Results saved successfully to {save_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results saved successfully to /content/drive/MyDrive/Daily_LLM_Summary_2026-04-17.csv


In [ ]:
print(headlines)


Headline 1: Apple withholds data in India antitrust case, watchdog sets final hearing - Reuters 
 Link: https://news.google.com/rss/articles/CBMi3gFBVV95cUxONGY5WnNXcDNfUlQ5MU4wRmdlLTlJSnFnTkVsanZwNFllWERVV1V5SlNJUkF1MENIcTFfUHlHZ3BENVN6ckhzWUNEeTM0OC1kUFNpUUpsblhmVjE4aTNXLWZZWHdHbE1LaWZhN1hVTkRkd0gzYk5tM2ZQZEJEX09zZk1lcVF2ejFzLV84WHVhbVM2RHJ0cThOdm9fRzdpUFJGeWtpMDhjN0Q5MUdlTHpDSG9ib0NmOFViUmVFeGRyQkZ6aEFrdWxfRERZemo1WmF3TGVLVk5NR2lLWktJb2c?oc=5 

Headline 2: Major 7.5-magnitude quake hits off Japan, triggers tsunami warnings - Reuters 
 Link: https://news.google.com/rss/articles/CBMivAFBVV95cUxQMUhrZE8yWnRQNzhsZG1na2tQTVlQYmlOcFR2elRyVEw3UDdpTktrbDJ1bUNsWm40dEYybDVzTWlrb2xnckg0ME5yalFQRDdYVE5QUVg1TWJmSkFmeVg3QmZ4Wk8xbDN0WElGOXgtU2hJemdRSVkxaklnb0w2SHRFbVdqVGlONVh3bW8yWFd1Uk11STZwczIySmU3dksxczNORUlUVDh5cUkzUThiQUkydTlzNVNqbTV3NmliRQ?oc=5 

Headline 3: China voices concern over US seizure of Iranian cargo ship, urges further talks - Reuters 
 Link: https://news.google.com/rss/articles

In [ ]:
print(f"Latest News Headlines: {headlines}")

Latest News Headlines: 
Headline 1: Apple withholds data in India antitrust case, watchdog sets final hearing - Reuters 
 Link: https://news.google.com/rss/articles/CBMi3gFBVV95cUxONGY5WnNXcDNfUlQ5MU4wRmdlLTlJSnFnTkVsanZwNFllWERVV1V5SlNJUkF1MENIcTFfUHlHZ3BENVN6ckhzWUNEeTM0OC1kUFNpUUpsblhmVjE4aTNXLWZZWHdHbE1LaWZhN1hVTkRkd0gzYk5tM2ZQZEJEX09zZk1lcVF2ejFzLV84WHVhbVM2RHJ0cThOdm9fRzdpUFJGeWtpMDhjN0Q5MUdlTHpDSG9ib0NmOFViUmVFeGRyQkZ6aEFrdWxfRERZemo1WmF3TGVLVk5NR2lLWktJb2c?oc=5 

Headline 2: Major 7.5-magnitude quake hits off Japan, triggers tsunami warnings - Reuters 
 Link: https://news.google.com/rss/articles/CBMivAFBVV95cUxQMUhrZE8yWnRQNzhsZG1na2tQTVlQYmlOcFR2elRyVEw3UDdpTktrbDJ1bUNsWm40dEYybDVzTWlrb2xnckg0ME5yalFQRDdYVE5QUVg1TWJmSkFmeVg3QmZ4Wk8xbDN0WElGOXgtU2hJemdRSVkxaklnb0w2SHRFbVdqVGlONVh3bW8yWFd1Uk11STZwczIySmU3dksxczNORUlUVDh5cUkzUThiQUkydTlzNVNqbTV3NmliRQ?oc=5 

Headline 3: China voices concern over US seizure of Iranian cargo ship, urges further talks - Reuters 
 Link: https://news.